# Análise SHAP - Explicabilidade do Modelo

Este notebook utiliza SHAP (SHapley Additive exPlanations) para interpretar os modelos de Machine Learning treinados no projeto de Análise de Risco de Crédito.

In [ ]:
# Importações
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Configuração de visualização
shap.initjs()

## 1. Carregamento dos Dados

In [ ]:
# Carregar dados processados
# Substitua pelo caminho correto do seu dataset
# df = pd.read_csv('../data/processed/credit_risk_clean.csv')

# Por enquanto, exemplo com dados sintéticos
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)
feature_names = [f'feature_{i}' for i in range(10)]
X = pd.DataFrame(X, columns=feature_names)

## 2. Treinamento do Modelo

In [ ]:
# Dividir dados
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Treinar modelo Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print(f'Acurácia no conjunto de teste: {model.score(X_test, y_test):.4f}')

## 3. Explicação Global com SHAP

Analisamos a importância global das features usando SHAP values.

In [ ]:
# Criar explainer SHAP
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Para classificação binária, pegar valores da classe positiva
if isinstance(shap_values, list):
    shap_values = shap_values[1]

### 3.1 Summary Plot (Beeswarm)

Mostra a distribuição dos SHAP values para cada feature.

In [ ]:
shap.summary_plot(shap_values, X_test, plot_type='bar')
plt.title('Importância Global das Features')
plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(shap_values, X_test)
plt.title('Distribuição dos SHAP Values')
plt.tight_layout()
plt.show()

## 4. Explicação Local - Casos Individuais

Analisamos como o modelo tomou decisões para casos específicos.

In [ ]:
# Waterfall plot para o primeiro caso de alto risco
idx = 0
shap.waterfall_plot(shap.Explanation(values=shap_values[idx], 
                                      base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
                                      data=X_test.iloc[idx],
                                      feature_names=X_test.columns.tolist()))

### 4.2 Force Plot

Visualização interativa da contribuição de cada feature.

In [ ]:
# Force plot para um caso individual
shap.force_plot(explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
                shap_values[idx],
                X_test.iloc[idx],
                matplotlib=True)
plt.tight_layout()
plt.show()

## 5. Dependência entre Features

Analisamos como o valor de uma feature afeta as predições.

In [ ]:
# Dependence plot para a feature mais importante
shap.dependence_plot(0, shap_values, X_test, feature_names=X_test.columns.tolist())
plt.tight_layout()
plt.show()

## Conclusões

Com SHAP conseguimos:
- Identificar as features mais importantes para as predições
- Entender como cada feature contribui para casos individuais
- Validar que o modelo está usando lógica apropriada
- Aumentar a confiança nas decisões do modelo